In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:

IMG_SIZE = (64, 64)
BATCH_SIZE = 32
EPOCHS = 20
DATASET_DIR= "/kaggle/input/datasets/grassknoted/asl-alphabet/asl_alphabet_train/asl_alphabet_train"

In [ ]:
#  FILE INDEXING & STRATIFIED SPLIT
def create_dataframe(data_dir):
    filepaths = []
    labels = []
    valid_extensions = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')
    
    for folder in sorted(os.listdir(data_dir)):
        folder_path = os.path.join(data_dir, folder)
        if os.path.isdir(folder_path):
            for file in os.listdir(folder_path):
                if file.endswith(valid_extensions):
                    filepaths.append(os.path.join(folder_path, file))
                    labels.append(folder)
                    
    df = pd.DataFrame({'filename': filepaths, 'label': labels})
    
    if len(df) == 0:
        raise ValueError(f"No image files found in '{data_dir}'. Verify dataset directory contents.")
        
    return df

df = create_dataframe(DATASET_DIR)

#  Split
train_df, val_df = train_test_split(
    df, 
    test_size=0.20, 
    random_state=42, 
    stratify=df['label']
)

# Reset indices 
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"[INFO] Dataset loaded: {len(df)} total images ({len(train_df)} train, {len(val_df)} validation).")



In [ ]:
#  DATA VISUALIZATION
def plot_distribution(df):
    plt.figure(figsize=(14, 4))
    sns.countplot(data=df, x='label', order=sorted(df['label'].unique()), palette="viridis")
    plt.xticks(rotation=45)
    plt.title("Class Distribution (Stratified Dataset)")
    plt.xlabel("Sign Language Category")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig("class_distribution.png")
    plt.show()

plot_distribution(df)

In [ ]:
#  PREPROCESSING & DATA GENERATORS
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=10,
    width_shift_range=0.08,
    height_shift_range=0.08,
    zoom_range=0.08,
    horizontal_flip=False  
)

val_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filename',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='filename',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
#  CNN ARCHITECTURE

def build_stable_cnn(input_shape=(64, 64, 3), num_classes=29):
    model = models.Sequential([
        
        layers.Conv2D(32, (3, 3), padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),

        
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),

      
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),

        
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    
   
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_stable_cnn(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), num_classes=len(train_generator.class_indices))
model.summary()

In [ ]:

# MODEL TRAINING

callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1, min_lr=1e-6)
]

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks
)

model.save("sign_language_cnn_stable.h5")


In [ ]:
model.save("sign_language_cnn_stable.keras")


In [ ]:
 # METRICS VISUALIZATION

def plot_metrics(history_obj):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy Plot
    ax1.plot(history_obj.history['accuracy'], label='Train Accuracy', color='blue', linewidth=2)
    ax1.plot(history_obj.history['val_accuracy'], label='Val Accuracy', color='orange', linewidth=2)
    ax1.set_title('Model Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True)

    # Loss Plot
    ax2.plot(history_obj.history['loss'], label='Train Loss', color='blue', linewidth=2)
    ax2.plot(history_obj.history['val_loss'], label='Val Loss', color='orange', linewidth=2)
    ax2.set_title('Model Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig("stable_accuracy_loss_graph.png")
    plt.show()

plot_metrics(history)